# 1.  GMS를 활용한 Embedding model 활용

- OpenAI client 및 문서 정보 정의​

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

GMS_KEY = os.getenv('GMS_KEY')
print(GMS_KEY)

In [ ]:
post_manuals = {
    "자바스크립트언어": "자바스크립트는 웹 개발에 필수적인 프로그래밍 언어입니다.",
    "일본관광시기": "일본은 벚꽃이 피는 봄이 관광하기 가장 좋은 시기입니다.",
    "파이썬언어": "파이썬 언어는 데이터분석과 기계학습에 효율적인 프로그래밍 언어입니다",
    "기계학습기초": "기계학습은 데이터를 활용하여 컴퓨터가 학습하도록 하는 기술입니다.",
    "스페인방문계절": "스페인은 날씨가 온화한 봄이나 가을에 방문하는 것이 이상적입니다."
}

POST_LEN = len(post_manuals)

In [ ]:
from openai import OpenAI

# OpenAI endpoint 설정
url = "https://gms.ssafy.io/gmsapi/api.openai.com/v1"
client = OpenAI(api_key=GMS_KEY, base_url=url)

# post_manuals에 있는 각 value를 벡터로 변환
response = client.embeddings.create(
    model="text-embedding-3-small",  # OpenAI Embedding 요청
    input=post_manuals.values(),
    encoding_format='float'
).data

print(response)

In [ ]:
# 결과에서 'embedding' 벡터만 추출
embedding_vectors = [item.embedding for item in response]

# 임베딩 차원 확인
# 문장이 각각 1536 차원 벡터으로 변환 되었음을 확인
print(len(embedding_vectors[0]),
      len(embedding_vectors[1]),
      len(embedding_vectors[2]),
      len(embedding_vectors[3]),
      len(embedding_vectors[4]),)

# 2. 문서 벡터 기반 코싸인 유사도 계산 및 결과 확인

In [ ]:
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(A, B):
    # 코사인 유사도 = (A 내적 B) / (A 길이 x B 길이)
    return dot(A, B) / (norm(A) * norm(B))


# 유사도 저장을 위한 리스트
similarities = [[0] * POST_LEN for _ in range(POST_LEN)]
# 문서(문장)들 간의 코사인 유사도 출력
for i in range(POST_LEN):
    for j in range(i+1, POST_LEN):
        similarity = cosine_similarity(embedding_vectors[i],
                                       embedding_vectors[j])
        print(f'문서 {i+1}과 문서 {j+1}의 유사도 : {similarity}')
        similarities[i][j] = similarity
        similarities[j][i] = similarity
        

In [ ]:
# scikit learn 에 있는 cosine_similarity 활용
# from sklearn.metrics.pairwise import cosine_similarity

# similarities = cosine_similarity(embedding_vectors, embedding_vectors)
# print(similarities)


In [ ]:
# 유사한 글 추천 함수
def recommendations(title):
    posts_titles = list(post_manuals)
    # 블로그 제목을 입력하면 해당 제목의 인덱스를 리턴받아 idx에 저장.
    if title in posts_titles :
        idx = posts_titles.index(title)
        similar_doc = similarities[idx]
    else :
        print("도서 정보가 존재하지 않습니다.")
        return []
    
    # 입력된 블로그와 내용(document embedding)이 유사한 글 3개 선정.
    sim_scores = list(enumerate(similar_doc)) # (index, 유사도) 튜플의 리스트
    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True) # 유사도 높은 순서로 sorting
    
    similar_posts_titles = []
    # 유사한 블로그 글 제목 출력
    for index, post_info in sim_scores:
        # 같은 글인 경우 제외
        if title == posts_titles[index]:
            continue

        if post_info > 0.3:     # 특정 유사도를 넘어가는 글을 추천
            title=posts_titles[index]
            similar_posts_titles.append(title)
        
    return similar_posts_titles


In [ ]:
results = recommendations('자바스크립트언어')
print(results)  # ['파이썬언어']

> recommendations 함수는 유사한 문서 데이터를 찾아오는 RAG의 Retrieval 의 기초적인 동작이 된다.
>
> 해당 함수의 결과를 LLM에 넣어 답변을 생성하도록 요청하는 함수를 추가 작성한다면 기초적인 RAG가 완성된다.

# [참고1] 임베딩 벡터 파일, 코싸인 유사도 파일 저장하고 불러쓰기

In [ ]:
import pickle

# pickle로 저장하기
# with open("embedding_vectors.pickle","wb") as f:
#     pickle.dump(embedding_vectors, f)

# pickle로 불러오기
# with open("embedding_vectors.pickle","rb") as f:
# 	embedding_vectors=pickle.load(f)

# pickle로 저장하기
# with open("similarities.pickle","wb") as f:
#     pickle.dump(similarities, f)

# pickle로 불러오기
# with open("similarities.pickle","rb") as f:
# 	similarities=pickle.load(f)